Binned data (e.g.:
-Geburtsland (Gruppen): Insgesamt_Bevoelkerung_Geburtsland_Gruppen_10km-Gitter + Germany/EU27/… bins.
-Staatsangehörigkeit (Gruppen): Insgesamt_Bevoelkerung_Staatsangehoerigkeit_Gruppen_10km-Gitter + Germany/EU27/… bins.
-Wohnungsfläche (10 m²-Intervalle): Insgesamt_Wohnungen_Flaeche_der_Wohnung_10m2_Intervalle_* + unter30…180+.
-Wohnungen nach Zimmerzahl: Insgesamt_Wohnungen_Wohnungen_nach_Zahl_der_Raeume_* + 1 Raum…7+.
etc.)
Processed similarly to ages

In [ ]:
# -*- coding: utf-8 -*-
"""
Generic hierarchical downscaling for NON-AGE categorical vectors.

Core idea (per topic):
  - Child HARD rows  : child topic total column (e.g., "Insgesamt_*") [optionally *_adj]
  - Parent HARD cols : parent per-category totals (sum over categories at parent level)
  - Trust-blended local -> parent shares (configurable w_min / t_pc) + damping alpha
  - Robust raking/IPF to satisfy both margins

Configure topics in build_topic_specs_for_level(), run 10km→1km and/or 1km→100m.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from tqdm import tqdm

# ---------------------------------------------------------------------
# 0) Trust blending (central, easy-to-tune)
# ---------------------------------------------------------------------

@dataclass
class TrustBlend:
    """
    Compute local trust weight w_i in [w_min, 1] from child row totals T_i.
    - w_min : baseline trust even for tiny totals
    - t_pc  : "people per category" target; when T_i ≈ K * t_pc, w_i ~ 1
              (K = number of categories in the topic)
    """
    w_min: float = 0.40
    t_pc: float  = 10.0

    def weights(self, totals: np.ndarray, n_categories: int) -> np.ndarray:
        cutoff = max(self.t_pc * max(n_categories, 1), 1e-9)
        base = np.minimum(1.0, totals / cutoff)
        return self.w_min + (1.0 - self.w_min) * base


# ---------------------------------------------------------------------
# 1) IPF / raking with hard margins
# ---------------------------------------------------------------------

def rake_to_margins(X: np.ndarray,
                    row_targets: np.ndarray,
                    col_targets: np.ndarray,
                    *, tol: float = 1e-10, max_iter: int = 50) -> None:
    """In-place IPF so rows -> row_targets and cols -> col_targets."""
    assert X.ndim == 2
    n, k = X.shape
    assert row_targets.shape == (n,)
    assert col_targets.shape == (k,)

    eps = tol
    for _ in range(max_iter):
        # columns
        col_sums = X.sum(axis=0)
        cscale = np.divide(col_targets, np.maximum(col_sums, eps))
        X *= cscale  # broadcast over rows

        # rows
        row_sums = X.sum(axis=1, keepdims=True)
        rscale = np.divide(row_targets[:, None], np.maximum(row_sums, eps))
        X *= rscale  # broadcast over cols

        if (np.allclose(X.sum(axis=0), col_targets, rtol=1e-9, atol=1e-9) and
            np.allclose(X.sum(axis=1), row_targets, rtol=1e-9, atol=1e-9)):
            break


# ---------------------------------------------------------------------
# 2) Child totals adjuster (scalar per parent group)
# ---------------------------------------------------------------------

def make_child_totals_adj(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    parent_adj_col: str,   # parent-level HARD total to hit (e.g. "Insgesamt_*_adj" or raw "Insgesamt_*")
    child_total_col: str,  # child-level "Insgesamt_*" to be scaled (writes *_adj)
    out_col: Optional[str] = None
) -> pd.Series:
    """
    For each parent id, multiply child totals by a scalar so
    sum(children)_adj == parent_adj. Returns the new child series (also added as column).
    If a group's child sum is 0 but parent_adj > 0, split equally across its children.
    """
    out_col = out_col or f"{child_total_col}_adj"

    pids_parent = parent_df[parent_id_col].astype(str).str.strip()
    pids_child  = child_df[child_parent_id_col].astype(str).str.strip()

    parent_target = (parent_df
        .assign(_pid=pids_parent)
        .set_index("_pid")[parent_adj_col]
        .astype(float)
        .reindex(pids_child.unique())
        .fillna(0.0))

    child_tot = pd.to_numeric(child_df[child_total_col], errors="coerce").fillna(0.0)
    group_sum = pd.Series(child_tot.values, index=pids_child).groupby(level=0).sum().astype(float)

    # scale factor per parent id
    scale = pd.Series(0.0, index=group_sum.index)
    mask_nz = group_sum > 0
    scale.loc[mask_nz] = parent_target.loc[mask_nz] / group_sum.loc[mask_nz]

    # broadcast
    s = pids_child.map(scale).fillna(0.0)
    adj = child_tot * s

    # degenerate: group_sum==0 & parent_target>0  -> equal split
    deg = (~mask_nz) & (parent_target > 0)
    if deg.any():
        counts = pids_child.value_counts().reindex(parent_target.index).fillna(0).astype(int)
        per_cap = (parent_target / counts.replace(0, np.nan)).fillna(0.0)
        use_deg = pids_child.isin(per_cap.index[deg])
        adj.loc[use_deg] = pids_child.map(per_cap)[use_deg].values

    child_df[out_col] = adj.values
    return child_df[out_col]


# ---------------------------------------------------------------------
# 3) Topic spec & downscaler
# ---------------------------------------------------------------------

@dataclass
class TopicSpec:
    """
    One categorical vector at a given level pair (parent→child).
    You must provide:
      - parent category columns (order matters),
      - child  category columns (same order, same length),
      - child HARD row-total column (usually the topic's "Insgesamt_*", ideally *_adj).
    Optionally:
      - alpha (damping for multiplicative scaling),
      - blend (TrustBlend) to compute row weights w_i.
    """
    name: str
    parent_cat_cols: List[str]
    child_cat_cols:  List[str]
    child_row_total_col: str
    alpha: float = 0.85
    blend: TrustBlend = field(default_factory=TrustBlend)


def _assert_topic_columns(parent_df, child_df, parent_id_col, child_parent_id_col, spec: TopicSpec) -> None:
    missing_parent = [c for c in spec.parent_cat_cols if c not in parent_df.columns]
    missing_child  = [c for c in spec.child_cat_cols  if c not in child_df.columns]
    if missing_parent:
        raise KeyError(f"[{spec.name}] Missing parent cols: {missing_parent[:5]}{'...' if len(missing_parent)>5 else ''}")
    if missing_child:
        raise KeyError(f"[{spec.name}] Missing child cols: {missing_child[:5]}{'...' if len(missing_child)>5 else ''}")
    if spec.child_row_total_col not in child_df.columns:
        raise KeyError(f"[{spec.name}] Missing child total col: {spec.child_row_total_col}")
    if len(spec.parent_cat_cols) != len(spec.child_cat_cols):
        raise ValueError(f"[{spec.name}] Category length mismatch parent({len(spec.parent_cat_cols)}) "
                         f"vs child({len(spec.child_cat_cols)})")
    # id columns
    if parent_id_col not in parent_df.columns:
        raise KeyError(f"[{spec.name}] Missing parent_id_col: {parent_id_col}")
    if child_parent_id_col not in child_df.columns:
        raise KeyError(f"[{spec.name}] Missing child_parent_id_col: {child_parent_id_col}")


def _apply_topic_trust_blend(
    cdf: pd.DataFrame,        # child slice for one parent
    X: np.ndarray,            # (n_child, K)
    child_totals: np.ndarray, # (n_child,)
    parent_shares: np.ndarray,# (K,)
    child_cat_cols: List[str],
    alpha: float,
    blend: TrustBlend,
    eps: float = 1e-9
) -> None:
    """
    For each category k:
      target_i = w_i * local_ik + (1 - w_i) * (parent_share_k * child_total_i)
      scale X[:, k] by (target/s) ** alpha, with protections & seeding.
    """
    n, K = X.shape
    assert K == len(child_cat_cols)
    w = blend.weights(child_totals, K)

    for k, col in enumerate(child_cat_cols):
        local = pd.to_numeric(cdf[col], errors="coerce").fillna(0.0).to_numpy(float)  # (n,)
        nat_exp = parent_shares[k] * child_totals
        target  = w * local + (1.0 - w) * nat_exp

        # seed first (if target>0 while current ≈ 0), then compute scaling
        s = X[:, k]
        need_seed = (s <= eps) & (target > 0)
        if need_seed.any():
            X[need_seed, k] = target[need_seed]
            s = X[:, k]  # refresh after seeding

        good = (s > eps) & np.isfinite(s)
        f = np.ones_like(s)
        f[good] = (np.maximum(target[good], 0.0) / s[good]) ** alpha
        np.clip(f, 1e-6, 1e6, out=f)
        X[:, k] *= f


def downscale_topic(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    spec: TopicSpec,
    inner_passes: int = 10,
    outer_iters: int = 2,
    rake_tol: float = 1e-10,
    rake_max_iter: int = 50,
    validate_row_tol: float = 2e-4,     # 0.02%
    verbose: bool = False
) -> pd.DataFrame:
    """
    Downscale one TopicSpec from parent -> child.
    Returns a DataFrame aligned to child_df index with the same category columns as spec.child_cat_cols.
    """
    # minimal config safety
    _assert_topic_columns(parent_df, child_df, parent_id_col, child_parent_id_col, spec)

    # normalize IDs inside (robust to standalone calls)
    parent_df = parent_df.copy()
    child_df  = child_df.copy()
    parent_df[parent_id_col]      = parent_df[parent_id_col].astype(str).str.strip()
    child_df[child_parent_id_col] = child_df[child_parent_id_col].astype(str).str.strip()

    K = len(spec.child_cat_cols)
    out = pd.DataFrame(index=child_df.index, columns=spec.child_cat_cols, dtype=float)

    # Pre-compute parent per-category vector per parent_id
    parent_map: Dict[object, np.ndarray] = {}
    for pid, psub in parent_df[[parent_id_col] + spec.parent_cat_cols].groupby(parent_id_col):
        parent_map[pid] = psub[spec.parent_cat_cols].astype(float).to_numpy().sum(axis=0)

    # iterate by parent group in child (direct frames)
    for pid, cdf in tqdm(child_df.groupby(child_parent_id_col, sort=False, group_keys=False),
                         disable=not verbose):
        idx = cdf.index
        p = parent_map.get(pid)

        if p is None or cdf.empty:
            out.loc[idx, :] = 0.0
            continue

        # HARD row totals (child)
        t = pd.to_numeric(cdf[spec.child_row_total_col], errors="coerce").fillna(0.0).to_numpy(float)
        n = t.size
        if n == 0:
            continue

        Psum = float(np.sum(p))
        if Psum <= 0:
            out.loc[idx, :] = 0.0
            continue

        parent_shares = p / Psum  # (K,)
        X = np.outer(t, parent_shares)  # (n, K)

        for _ in range(outer_iters):
            for _inner in range(inner_passes):
                _apply_topic_trust_blend(
                    cdf, X, t, parent_shares, spec.child_cat_cols,
                    alpha=spec.alpha, blend=spec.blend
                )
                # re-project to hard child totals
                rs = X.sum(axis=1, keepdims=True)
                np.divide(t[:, None], np.maximum(rs, 1e-12), out=rs)
                X *= rs

            # hard parent per-category via IPF
            rake_to_margins(X, row_targets=t, col_targets=p, tol=rake_tol, max_iter=rake_max_iter)

        out.loc[idx, :] = X

        # validations
        calc_rows = X.sum(axis=1)
        rel = np.abs(calc_rows - t) / np.maximum(t, 1.0)
        if rel.max(initial=0.0) > validate_row_tol:
            raise AssertionError(
                f"[{spec.name}] Row totals dev > {validate_row_tol:.2e} for parent {pid}. "
                f"max rel.err={rel.max():.3e}"
            )
        if not np.allclose(X.sum(axis=0), p, rtol=0, atol=1e-5):
            if not np.allclose(X.sum(axis=0), p, rtol=0, atol=0.1):
                raise AssertionError(f"[{spec.name}] Col totals != parent per-category for parent {pid}.")

    return out


# ---------------------------------------------------------------------
# 4) Helpers for level-specific column names (simple suffix swap)
# ---------------------------------------------------------------------

def levelize(cols_10km: List[str], level: str) -> List[str]:
    """
    Replace the trailing suffix *_10km-Gitter with *_{level}-Gitter
    level ∈ {"10km","1km","100m"}
    """
    from_str = "_10km-Gitter"
    to_str   = f"_{level}-Gitter"
    return [c.replace(from_str, to_str) for c in cols_10km]


# ---------------------------------------------------------------------
# 5) Example configuration for PRIORITY topics
# ---------------------------------------------------------------------

BLEND_STRONG = TrustBlend(w_min=0.50, t_pc=10.0)
BLEND_STD    = TrustBlend(w_min=0.40, t_pc=10.0) # This is the default and similar to what we did with population (ages)
BLEND_WEAK   = TrustBlend(w_min=0.30, t_pc=30.0)

def build_topic_specs_for_level(level: str) -> List[TopicSpec]:
    """
    Returns TopicSpec list for a given child level, assuming parent is the next coarser level:
      - for level="1km": parent is 10km
      - for level="100m": parent is 1km
    """
    # --- Familienstand (population by marital status) ---
    fam_tot_10 = "Insgesamt_Bevoelkerung_Familienstand_10km-Gitter"
    fam_cats_10 = [
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ]

    # --- Energieträger (population; NOT buildings-by-energy) ---
    et_tot_10 = "Insgesamt_Energietraeger_Energietraeger_10km-Gitter"
    et_cats_10 = [
        "Gas_Energietraeger_10km-Gitter",
        "Heizoel_Energietraeger_10km-Gitter",
        "Holz_Holzpellets_Energietraeger_10km-Gitter",
        "Biomasse_Biogas_Energietraeger_10km-Gitter",
        "Solar_Geothermie_Waermepumpen_Energietraeger_10km-Gitter",
        "Strom_Energietraeger_10km-Gitter",
        "Kohle_Energietraeger_10km-Gitter",
        "Fernwaerme_Energietraeger_10km-Gitter",
        "kein_Energietraeger_Energietraeger_10km-Gitter",
    ]

    # --- Heizungsart (Gebäude nach überwiegender Heizungsart) ---
    hz_tot_10 = "Insgesamt_Heizungsart_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter"
    hz_cats_10 = [
        "Fernheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Etagenheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Blockheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Zentralheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Einzel_Mehrraumoefen_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "keine_Heizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
    ]

    # --- Haushaltsgröße (size of private household) ---
    hh_tot_10 = "Insgesamt_Haushalte_Groesse_des_privaten_Haushalts_10km-Gitter"
    hh_cats_10 = [
        "1_Person_Groesse_des_privaten_Haushalts_10km-Gitter",
        "2_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "3_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "4_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "5_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "6_Personen_und_mehr_Groesse_des_privaten_Haushalts_10km-Gitter",
    ]

    # --- Lebensform (private HH life form) ---
    lf_tot_10 = "Insgesamt_Haushalte_Typ_priv_HH_Lebensform_10km-Gitter"
    lf_cats_10 = [
        "EinpersHH_SingleHH_Typ_priv_HH_Lebensform_10km-Gitter",
        "Ehepaare_Typ_priv_HH_Lebensform_10km-Gitter",
        "EingetrLebensp_Typ_priv_HH_Lebensform_10km-Gitter",
        "NichtehelLebensg_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzMuetter_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzVaeter_Typ_priv_HH_Lebensform_10km-Gitter",
        "MehrpersHHohneKernfam_Typ_priv_HH_Lebensform_10km-Gitter",
    ]

    # --- Räume (Wohnungen nach Zahl der Räume) ---
    rm_tot_10 = "Insgesamt_Wohnungen_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter"
    rm_cats_10 = [
        "1Raum_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "2Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "3Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "4Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "5Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "6Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "7undmehrRaeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
    ]

    # --- Wohnungsfläche (10 m² intervals) ---
    fl_tot_10 = "Insgesamt_Wohnungen_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter"
    fl_cats_10 = [
        "unter30_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "30bis39_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "40bis49_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "50bis59_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "60bis69_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "70bis79_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "80bis89_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "90bis99_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "100bis109_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "110bis119_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "120bis129_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "130bis139_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "140bis149_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "150bis159_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "160bis169_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "170bis179_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "180undmehr_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
    ]

    # --- Geburtsland (groups) ---
    gl_tot_10 = "Insgesamt_Bevoelkerung_Geburtsland_Gruppen_10km-Gitter"
    gl_cats_10 = [
        "Deutschland_Geburtsland_Gruppen_10km-Gitter",
        "Ausland_Sonstige_Geburtsland_Gruppen_10km-Gitter",
        "EU27_Land_Geburtsland_Gruppen_10km-Gitter",
        "Sonstiges_Europa_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Welt_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Geburtsland_Gruppen_10km-Gitter",
    ]

    def topic(name, tot_10, cats_10, alpha, blend):
        if level == "1km":
            parent_cols = cats_10
            child_cols  = levelize(cats_10, "1km")
            child_total = tot_10.replace("_10km-Gitter", "_1km-Gitter")
        elif level == "100m":
            parent_cols = levelize(cats_10, "1km")
            child_cols  = levelize(cats_10, "100m")
            child_total = tot_10.replace("_10km-Gitter", "_100m-Gitter")  # correct for 100m
        else:
            raise ValueError(f"Unknown level: {level}")

        return TopicSpec(
            name=name,
            parent_cat_cols=parent_cols,
            child_cat_cols=child_cols,
            child_row_total_col=child_total,
            alpha=alpha,
            blend=blend,
        )

    specs = [
        topic("Familienstand",    fam_tot_10, fam_cats_10, alpha=0.90, blend=BLEND_STD),
        topic("Energietraeger",   et_tot_10,  et_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Heizungsart",      hz_tot_10,  hz_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Haushaltsgroesse", hh_tot_10,  hh_cats_10,  alpha=0.90, blend=BLEND_STD),
        topic("Lebensform",       lf_tot_10,  lf_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Raeume",           rm_tot_10,  rm_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Wohnflaeche",      fl_tot_10,  fl_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Geburtsland",      gl_tot_10,  gl_cats_10,  alpha=0.85, blend=BLEND_STD),
    ]
    return specs


# ---------------------------------------------------------------------
# 5.5) Shim: create *_adj child totals for ALL topics and flip specs
# ---------------------------------------------------------------------

def _parent_total_name_for_child_total(child_total_col: str) -> str:
    """Swap suffix to get the corresponding parent level total name."""
    if child_total_col.endswith("_1km-Gitter"):
        return child_total_col.replace("_1km-Gitter", "_10km-Gitter")
    elif child_total_col.endswith("_100m-Gitter"):
        return child_total_col.replace("_100m-Gitter", "_1km-Gitter")
    else:
        # Already 10km or unknown; return as-is (caller should pass correct levels)
        return child_total_col

def apply_adj_for_all_topics(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    specs: List[TopicSpec],
    verbose: bool = True,
) -> List[TopicSpec]:
    """For each TopicSpec, create a *_adj child row-total and switch the spec to use it."""
    specs_out: List[TopicSpec] = []
    for spec in specs:
        total_col = spec.child_row_total_col
        adj_col = f"{total_col}_adj"
        if adj_col not in child_df.columns:
            parent_total = _parent_total_name_for_child_total(total_col)
            make_child_totals_adj(
                parent_df=parent_df,
                child_df=child_df,
                parent_id_col=parent_id_col,
                child_parent_id_col=child_parent_id_col,
                parent_adj_col=parent_total,
                child_total_col=total_col,
                out_col=adj_col,
            )
            if verbose:
                print(f"[adj] Created {adj_col} for topic '{spec.name}'")
        # flip the spec to use *_adj
        specs_out.append(TopicSpec(
            name=spec.name,
            parent_cat_cols=spec.parent_cat_cols,
            child_cat_cols=spec.child_cat_cols,
            child_row_total_col=adj_col,
            alpha=spec.alpha,
            blend=spec.blend,
        ))
    return specs_out


# ---------------------------------------------------------------------
# 6) Example run (wire it like you did for ages)
# ---------------------------------------------------------------------

if __name__ == "__main__":
    # Paths
    # We use the most recently edited/produced file in each cell size - but in the end 100m is what matters
    PATH_10  = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df10_with_single_years.pickle"  # parent for 1km, must contain the topic columns at 10km
    PATH_1   = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df1_with_single_years_100_4.pickle"
    PATH_100 = r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gender_backfilled_100_4.parquet" # we gendered and backfilled only 100m.

    OUT_1    = r"C:\Users\petre\PycharmProjects\cleancensus\with_other_binned_data\cells_1km_with_binneds_100_4.parquet"
    OUT_100  = r"C:\Users\petre\PycharmProjects\cleancensus\with_other_binned_data\cells_100m_with_gender_backf_binneds_100_4.parquet"

    # Load, reset index so IDs are normal cols
    df10  = pd.read_pickle(PATH_10).reset_index(drop=False)
    df1   = pd.read_pickle(PATH_1).reset_index(drop=False)
    df100 = pd.read_parquet(PATH_100).reset_index(drop=False)

    # Clean NaNs/±inf
    for df in (df10, df1, df100):
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.fillna(0, inplace=True)

    # Normalize IDs
    df10["GITTER_ID_10km"] = df10["GITTER_ID_10km"].astype(str).str.strip()
    df1["GITTER_ID_10km"]  = df1["GITTER_ID_10km"].astype(str).str.strip()
    df1["GITTER_ID_1km"]   = df1["GITTER_ID_1km"].astype(str).str.strip()
    df100["GITTER_ID_1km"] = df100["GITTER_ID_1km"].astype(str).str.strip()

    # -------------------------
    # 10km → 1km
    # -------------------------
    topics_1km = build_topic_specs_for_level("1km")

    # Create *_adj for ALL topics at 1km and flip specs to use them
    topics_1km = apply_adj_for_all_topics(
        parent_df=df10, child_df=df1,
        parent_id_col="GITTER_ID_10km", child_parent_id_col="GITTER_ID_10km",
        specs=topics_1km, verbose=True
    )

    for spec in tqdm(topics_1km, f"Topics 1km"):
        res = downscale_topic(
            parent_df=df10, child_df=df1,
            parent_id_col="GITTER_ID_10km",
            child_parent_id_col="GITTER_ID_10km",
            spec=spec,
            inner_passes=10, outer_iters=2,
            rake_tol=1e-10, rake_max_iter=50,
            validate_row_tol=2e-4,  # 0.02%
            verbose=False
        )
        # attach result
        for c in spec.child_cat_cols:
            df1[c] = res[c].values

    df1.to_parquet(OUT_1, index=False)

    # -------------------------
    # 1km → 100m
    # -------------------------
    topics_100m = build_topic_specs_for_level("100m")

    # link coverage flags (optional)
    p_1km = set(df1["GITTER_ID_1km"].unique())
    c_100 = set(df100["GITTER_ID_1km"].unique())
    df1["has_no_children"] = ~df1["GITTER_ID_1km"].isin(c_100)
    df100["is_orphan"]     = ~df100["GITTER_ID_1km"].isin(p_1km)

    # subset to valid links for the run, then merge back
    df1_ok   = df1.loc[~df1["has_no_children"]].copy()
    df100_ok = df100.loc[~df100["is_orphan"]].copy()

    # Create *_adj for ALL topics at 100m and flip specs to use them
    topics_100m = apply_adj_for_all_topics(
        parent_df=df1_ok, child_df=df100_ok,
        parent_id_col="GITTER_ID_1km", child_parent_id_col="GITTER_ID_1km",
        specs=topics_100m, verbose=True
    )

    for spec in tqdm(topics_100m, f"Topics 100m"):
        res = downscale_topic(
            parent_df=df1_ok, child_df=df100_ok,
            parent_id_col="GITTER_ID_1km",
            child_parent_id_col="GITTER_ID_1km",
            spec=spec,
            inner_passes=10, outer_iters=2,
            rake_tol=1e-10, rake_max_iter=50,
            validate_row_tol=2e-4,
            verbose=False
        )
        for c in spec.child_cat_cols:
            # attach only to the clean subset, then backfill full df100
            df100.loc[df100_ok.index, c] = res[c].values

    df100.to_parquet(OUT_100, index=False)

    # quick echoes
    def _echo(df, cols, total_col, tag):
        if total_col not in df.columns:
            print(f"[{tag}] skipped: total column '{total_col}' not found.")
            return
        calc = df[cols].sum(axis=1).to_numpy(float)
        given = pd.to_numeric(df[total_col], errors="coerce").fillna(0.0).to_numpy(float)
        rel = np.abs(calc - given) / np.maximum(given, 1.0)
        print(f"[{tag}] rows={len(df):,} | max row rel.err={rel.max():.3e} | mean={rel.mean():.3e}")

    # Echo (Familienstand)
    fam1_cols = levelize([
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ], "1km")
    _echo(df1, fam1_cols,
          "Insgesamt_Bevoelkerung_Familienstand_1km-Gitter_adj",  # using *_adj
          "1km Familienstand")

    fam100_cols = levelize([
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ], "100m")
    _echo(df100, fam100_cols,
          "Insgesamt_Bevoelkerung_Familienstand_100m-Gitter_adj",  # using *_adj
          "100m Familienstand")


In [2]:
# -*- coding: utf-8 -*-
"""
Generic hierarchical downscaling for NON-AGE categorical vectors.

Core idea (per topic):
  - Child HARD rows  : child topic total column (e.g., "Insgesamt_*") [optionally *_adj]
  - Parent HARD cols : parent per-category totals (sum over categories at parent level)
  - Trust-blended local -> parent shares (configurable w_min / t_pc) + damping alpha
  - Robust raking/IPF to satisfy both margins

Configure topics in build_topic_specs_for_level(), run 10km→1km and/or 1km→100m.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from tqdm import tqdm

# for memory-safe full-output writing at 100m
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# ---------------------------------------------------------------------
# 0) Trust blending (central, easy-to-tune)
# ---------------------------------------------------------------------

@dataclass
class TrustBlend:
    """
    Compute local trust weight w_i in [w_min, 1] from child row totals T_i.
    - w_min : baseline trust even for tiny totals
    - t_pc  : "people per category" target; when T_i ≈ K * t_pc, w_i ~ 1
              (K = number of categories in the topic)
    """
    w_min: float = 0.40
    t_pc: float  = 10.0

    def weights(self, totals: np.ndarray, n_categories: int) -> np.ndarray:
        cutoff = max(self.t_pc * max(n_categories, 1), 1e-9)
        base = np.minimum(1.0, totals / cutoff)
        return self.w_min + (1.0 - self.w_min) * base


# ---------------------------------------------------------------------
# 1) IPF / raking with hard margins
# ---------------------------------------------------------------------

def rake_to_margins(X: np.ndarray,
                    row_targets: np.ndarray,
                    col_targets: np.ndarray,
                    *, tol: float = 1e-10, max_iter: int = 50) -> None:
    """In-place IPF so rows -> row_targets and cols -> col_targets."""
    assert X.ndim == 2
    n, k = X.shape
    assert row_targets.shape == (n,)
    assert col_targets.shape == (k,)

    eps = tol
    for _ in range(max_iter):
        # columns
        col_sums = X.sum(axis=0)
        cscale = np.divide(col_targets, np.maximum(col_sums, eps))
        X *= cscale  # broadcast over rows

        # rows
        row_sums = X.sum(axis=1, keepdims=True)
        rscale = np.divide(row_targets[:, None], np.maximum(row_sums, eps))
        X *= rscale  # broadcast over cols

        if (np.allclose(X.sum(axis=0), col_targets, rtol=1e-9, atol=1e-9) and
            np.allclose(X.sum(axis=1), row_targets, rtol=1e-9, atol=1e-9)):
            break


# ---------------------------------------------------------------------
# 2) Child totals adjuster (scalar per parent group)
# ---------------------------------------------------------------------

def make_child_totals_adj(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    parent_adj_col: str,   # parent-level HARD total to hit (e.g. "Insgesamt_*_adj" or raw "Insgesamt_*")
    child_total_col: str,  # child-level "Insgesamt_*" to be scaled (writes *_adj)
    out_col: Optional[str] = None
) -> pd.Series:
    """
    For each parent id, multiply child totals by a scalar so
    sum(children)_adj == parent_adj. Returns the new child series (also added as column).
    If a group's child sum is 0 but parent_adj > 0, split equally across its children.
    """
    out_col = out_col or f"{child_total_col}_adj"

    pids_parent = parent_df[parent_id_col].astype(str).str.strip()
    pids_child  = child_df[child_parent_id_col].astype(str).str.strip()

    parent_target = (parent_df
        .assign(_pid=pids_parent)
        .set_index("_pid")[parent_adj_col]
        .astype(float)
        .reindex(pids_child.unique())
        .fillna(0.0))

    child_tot = pd.to_numeric(child_df[child_total_col], errors="coerce").fillna(0.0)
    group_sum = pd.Series(child_tot.values, index=pids_child).groupby(level=0).sum().astype(float)

    scale = pd.Series(0.0, index=group_sum.index)
    mask_nz = group_sum > 0
    scale.loc[mask_nz] = parent_target.loc[mask_nz] / group_sum.loc[mask_nz]

    s = pids_child.map(scale).fillna(0.0)
    adj = child_tot * s

    # degenerate: group_sum==0 & parent_target>0  -> equal split
    deg = (~mask_nz) & (parent_target > 0)
    if deg.any():
        counts = pids_child.value_counts().reindex(parent_target.index).fillna(0).astype(int)
        per_cap = (parent_target / counts.replace(0, np.nan)).fillna(0.0)
        use_deg = pids_child.isin(per_cap.index[deg])
        adj.loc[use_deg] = pids_child.map(per_cap)[use_deg].values

    child_df[out_col] = adj.values
    return child_df[out_col]


# ---------------------------------------------------------------------
# 3) Topic spec & downscaler
# ---------------------------------------------------------------------

@dataclass
class TopicSpec:
    """
    One categorical vector at a given level pair (parent→child).
    Provide:
      - parent category columns (order matters),
      - child  category columns (same order, same length),
      - child HARD row-total column (usually "Insgesamt_*", ideally *_adj).
    Optional:
      - alpha (damping),
      - blend (TrustBlend) for row weights w_i.
    """
    name: str
    parent_cat_cols: List[str]
    child_cat_cols:  List[str]
    child_row_total_col: str
    alpha: float = 0.85
    blend: TrustBlend = field(default_factory=TrustBlend)


def _assert_topic_columns(parent_df, child_df, parent_id_col, child_parent_id_col, spec: TopicSpec) -> None:
    missing_parent = [c for c in spec.parent_cat_cols if c not in parent_df.columns]
    missing_child  = [c for c in spec.child_cat_cols  if c not in child_df.columns]
    if missing_parent:
        raise KeyError(f"[{spec.name}] Missing parent cols: {missing_parent[:5]}{'...' if len(missing_parent)>5 else ''}")
    if missing_child:
        raise KeyError(f"[{spec.name}] Missing child cols: {missing_child[:5]}{'...' if len(missing_child)>5 else ''}")
    if spec.child_row_total_col not in child_df.columns:
        raise KeyError(f"[{spec.name}] Missing child total col: {spec.child_row_total_col}")
    if len(spec.parent_cat_cols) != len(spec.child_cat_cols):
        raise ValueError(f"[{spec.name}] Category length mismatch parent({len(spec.parent_cat_cols)}) "
                         f"vs child({len(spec.child_cat_cols)})")
    if parent_id_col not in parent_df.columns:
        raise KeyError(f"[{spec.name}] Missing parent_id_col: {parent_id_col}")
    if child_parent_id_col not in child_df.columns:
        raise KeyError(f"[{spec.name}] Missing child_parent_id_col: {child_parent_id_col}")


def _apply_topic_trust_blend(
    cdf: pd.DataFrame,
    X: np.ndarray,
    child_totals: np.ndarray,
    parent_shares: np.ndarray,
    child_cat_cols: List[str],
    alpha: float,
    blend: TrustBlend,
    eps: float = 1e-9
) -> None:
    """
    For each category k:
      target_i = w_i * local_ik + (1 - w_i) * (parent_share_k * child_total_i)
      Then scale X[:, k] by (target/s) ** alpha, with protections & seeding.
    """
    n, K = X.shape
    assert K == len(child_cat_cols)
    w = blend.weights(child_totals, K)

    for k, col in enumerate(child_cat_cols):
        local = pd.to_numeric(cdf[col], errors="coerce").fillna(0.0).to_numpy(float)
        nat_exp = parent_shares[k] * child_totals
        target  = w * local + (1.0 - w) * nat_exp

        s = X[:, k]
        need_seed = (s <= eps) & (target > 0)
        if need_seed.any():
            X[need_seed, k] = target[need_seed]
            s = X[:, k]

        good = (s > eps) & np.isfinite(s)
        f = np.ones_like(s)
        f[good] = (np.maximum(target[good], 0.0) / s[good]) ** alpha
        np.clip(f, 1e-6, 1e6, out=f)
        X[:, k] *= f


def downscale_topic(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    spec: TopicSpec,
    inner_passes: int = 10,
    outer_iters: int = 2,
    rake_tol: float = 1e-10,
    rake_max_iter: int = 50,
    validate_row_tol: float = 2e-4,
    verbose: bool = False
) -> pd.DataFrame:
    """Downscale one TopicSpec from parent -> child."""
    _assert_topic_columns(parent_df, child_df, parent_id_col, child_parent_id_col, spec)

    parent_df = parent_df.copy()
    child_df  = child_df.copy()
    parent_df[parent_id_col]      = parent_df[parent_id_col].astype(str).str.strip()
    child_df[child_parent_id_col] = child_df[child_parent_id_col].astype(str).str.strip()

    K = len(spec.child_cat_cols)
    out = pd.DataFrame(index=child_df.index, columns=spec.child_cat_cols, dtype=float)

    parent_map: Dict[object, np.ndarray] = {}
    for pid, psub in parent_df[[parent_id_col] + spec.parent_cat_cols].groupby(parent_id_col):
        parent_map[pid] = psub[spec.parent_cat_cols].astype(float).to_numpy().sum(axis=0)

    for pid, cdf in tqdm(child_df.groupby(child_parent_id_col, sort=False, group_keys=False),
                         disable=not verbose):
        idx = cdf.index
        p = parent_map.get(pid)

        if p is None or cdf.empty:
            out.loc[idx, :] = 0.0
            continue

        t = pd.to_numeric(cdf[spec.child_row_total_col], errors="coerce").fillna(0.0).to_numpy(float)
        if t.size == 0:
            continue

        Psum = float(np.sum(p))
        if Psum <= 0:
            out.loc[idx, :] = 0.0
            continue

        parent_shares = p / Psum
        X = np.outer(t, parent_shares)

        for _ in range(outer_iters):
            for _inner in range(inner_passes):
                _apply_topic_trust_blend(
                    cdf, X, t, parent_shares, spec.child_cat_cols,
                    alpha=spec.alpha, blend=spec.blend
                )
                rs = X.sum(axis=1, keepdims=True)
                np.divide(t[:, None], np.maximum(rs, 1e-12), out=rs)
                X *= rs

            rake_to_margins(X, row_targets=t, col_targets=p, tol=rake_tol, max_iter=rake_max_iter)

        out.loc[idx, :] = X

        calc_rows = X.sum(axis=1)
        rel = np.abs(calc_rows - t) / np.maximum(t, 1.0)
        if rel.max(initial=0.0) > validate_row_tol:
            raise AssertionError(
                f"[{spec.name}] Row totals dev > {validate_row_tol:.2e} for parent {pid}. "
                f"max rel.err={rel.max():.3e}"
            )
        if not np.allclose(X.sum(axis=0), p, rtol=0, atol=1e-2):
            diff = X.sum(axis=0) - p
            rel = np.divide(diff, np.maximum(np.abs(p), 1e-12))
            print("\n\n[DEBUG mismatch]")
            print(f"Parent ID: {pid}")
            print("Parent totals (p):")
            print(p)
            print("Child sums (X.sum(axis=0)):")
            print(X.sum(axis=0))
            print("Difference (child - parent):")
            print(diff)
            print("Relative difference:")
            print(rel)
            if not np.allclose(X.sum(axis=0), p, rtol=0, atol=1): # May get close for large or many child cells
                raise AssertionError(f"[{spec.name}] Col totals != parent per-category for parent {pid}.")
    return out


# ---------------------------------------------------------------------
# 4) Helpers for level-specific column names (simple suffix swap)
# ---------------------------------------------------------------------

def levelize(cols_10km: List[str], level: str) -> List[str]:
    """
    Replace the trailing suffix *_10km-Gitter with *_{level}-Gitter
    level ∈ {"10km","1km","100m"}
    """
    from_str = "_10km-Gitter"
    to_str   = f"_{level}-Gitter"
    return [c.replace(from_str, to_str) for c in cols_10km]

# ---------------------------------------------------------------------
# 4.5) Normalize parent category vectors to their parent totals
# ---------------------------------------------------------------------

def _make_topic_prior_shares(df, cols):
    cols = [c for c in cols if c in df.columns]   # guard, though you already do this upstream
    if not cols:
        return np.full(1, 1.0)  # or return a uniform over K when you know K; see note below

    vals = df.loc[:, cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    w = vals.sum(axis=0).to_numpy(dtype=float)
    s = float(w.sum())
    if s > 0:
        return w / s
    return np.full(len(cols), 1.0 / len(cols), dtype=float)


def normalize_parent_categories_for_specs(
    parent_df: pd.DataFrame,
    *,
    specs: List[TopicSpec],
    child_level: str,            # '1km'
    cap_factor: float = 10.0,    # optional: flag overly large corrections
    verbose: bool = True,
) -> None:
    """
    For each TopicSpec:
      - compute a global prior over its parent category columns,
      - per parent row, scale cats so sum(cats) == parent total,
      - if sum(cats)==0 < and total>0, inject prior * total,
      - if total==0, set cats to 0.
    Operates IN-PLACE on parent_df.
    """
    for spec in specs:
        cat_cols = [c for c in spec.parent_cat_cols if c in parent_df.columns]
        if not cat_cols:
            if verbose:
                print(f"[norm] skip '{spec.name}': no parent category cols present")
            continue

        if child_level == "1km":
            tot_col = spec.child_row_total_col.replace("_1km-Gitter", "_10km-Gitter")
        # elif child_level == "100m":
        #     tot_col = spec.child_row_total_col.replace("_100m-Gitter", "_1km-Gitter") # NOT to be used at that level. Just for 10km init.
        else:
            raise ValueError(f"Unknown child_level: {child_level}")

        if tot_col not in parent_df.columns:
            if verbose:
                print(f"[norm] skip '{spec.name}': parent total '{tot_col}' not found")
            continue

        # Global prior shares for this topic
        prior = _make_topic_prior_shares(parent_df, cat_cols)  # (K,)

        # Numeric frames
        C = parent_df[cat_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)  # (N,K)
        T = pd.to_numeric(parent_df[tot_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)         # (N,)

        S = C.sum(axis=1)  # (N,)
        N, K = C.shape

        # Cases
        # 1) total == 0 -> set row to 0
        zero_tot = T <= 0
        if np.any(zero_tot):
            C[zero_tot, :] = 0.0

        # 2) sum(cats) == 0 & total > 0 -> inject prior * total
        need_inject = (S <= 0) & (T > 0)
        if np.any(need_inject):
            C[need_inject, :] = T[need_inject, None] * prior[None, :]

        # 3) baseline scaling for all rows with S > 0
        S = C.sum(axis=1)  # recompute after injection
        good = S > 0
        f = np.ones_like(S)
        f[good] = T[good] / S[good]

        # Optional: flag extreme factors
        if verbose:
            big = np.abs(f[good]) > cap_factor
            if np.any(big):
                n_big = int(np.sum(big))
                print(f"[norm] '{spec.name}' warning: {n_big} rows have |scale| > {cap_factor} (max={float(np.max(np.abs(f[good]))):.2f})")

        C *= f[:, None]

        # Write back
        parent_df.loc[:, cat_cols] = C
        if verbose:
            # cheap audit: check new sums vs totals
            new_s = C.sum(axis=1)
            err = np.abs(new_s - T) / np.maximum(T, 1.0)
            print(f"[norm] '{spec.name}' rows={len(err):,} | max rel.err={float(err.max()):.2e} | mean={float(err.mean()):.2e}")


# ---------------------------------------------------------------------
# 5) Configuration for PRIORITY topics
# ---------------------------------------------------------------------

BLEND_STRONG = TrustBlend(w_min=0.50, t_pc=10.0)
BLEND_STD    = TrustBlend(w_min=0.40, t_pc=10.0)
BLEND_WEAK   = TrustBlend(w_min=0.30, t_pc=30.0)

def build_topic_specs_for_level(level: str) -> List[TopicSpec]:
    """
    Returns TopicSpec list for a given child level, assuming parent is the next coarser level:
      - for level="1km": parent is 10km
      - for level="100m": parent is 1km
    """
    # --- Familienstand (population by marital status) ---
    fam_tot_10 = "Insgesamt_Bevoelkerung_Familienstand_10km-Gitter"
    fam_cats_10 = [
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ]

    # --- Energieträger (population; NOT buildings-by-energy) ---
    et_tot_10 = "Insgesamt_Energietraeger_Energietraeger_10km-Gitter"
    et_cats_10 = [
        "Gas_Energietraeger_10km-Gitter",
        "Heizoel_Energietraeger_10km-Gitter",
        "Holz_Holzpellets_Energietraeger_10km-Gitter",
        "Biomasse_Biogas_Energietraeger_10km-Gitter",
        "Solar_Geothermie_Waermepumpen_Energietraeger_10km-Gitter",
        "Strom_Energietraeger_10km-Gitter",
        "Kohle_Energietraeger_10km-Gitter",
        "Fernwaerme_Energietraeger_10km-Gitter",
        "kein_Energietraeger_Energietraeger_10km-Gitter",
    ]

    # --- Heizungsart (Gebäude nach überwiegender Heizungsart) ---
    hz_tot_10 = "Insgesamt_Heizungsart_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter"
    hz_cats_10 = [
        "Fernheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Etagenheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Blockheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Zentralheizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "Einzel_Mehrraumoefen_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
        "keine_Heizung_Gebaeude_nach_ueberwiegender_Heizungsart_10km-Gitter",
    ]

    # --- Haushaltsgröße (size of private household) ---
    hh_tot_10 = "Insgesamt_Haushalte_Groesse_des_privaten_Haushalts_10km-Gitter"
    hh_cats_10 = [
        "1_Person_Groesse_des_privaten_Haushalts_10km-Gitter",
        "2_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "3_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "4_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "5_Personen_Groesse_des_privaten_Haushalts_10km-Gitter",
        "6_Personen_und_mehr_Groesse_des_privaten_Haushalts_10km-Gitter",
    ]

    # --- Lebensform (private HH life form) ---
    lf_tot_10 = "Insgesamt_Haushalte_Typ_priv_HH_Lebensform_10km-Gitter"
    lf_cats_10 = [
        "EinpersHH_SingleHH_Typ_priv_HH_Lebensform_10km-Gitter",
        "Ehepaare_Typ_priv_HH_Lebensform_10km-Gitter",
        "EingetrLebensp_Typ_priv_HH_Lebensform_10km-Gitter",
        "NichtehelLebensg_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzMuetter_Typ_priv_HH_Lebensform_10km-Gitter",
        "AlleinerzVaeter_Typ_priv_HH_Lebensform_10km-Gitter",
        "MehrpersHHohneKernfam_Typ_priv_HH_Lebensform_10km-Gitter",
    ]

    # --- Räume (Wohnungen nach Zahl der Räume) ---
    rm_tot_10 = "Insgesamt_Wohnungen_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter"
    rm_cats_10 = [
        "1Raum_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "2Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "3Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "4Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "5Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "6Raeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
        "7undmehrRaeume_Wohnungen_nach_Zahl_der_Raeume_10km-Gitter",
    ]

    # --- Wohnungsfläche (10 m² intervals) ---
    fl_tot_10 = "Insgesamt_Wohnungen_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter"
    fl_cats_10 = [
        "unter30_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "30bis39_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "40bis49_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "50bis59_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "60bis69_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "70bis79_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "80bis89_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "90bis99_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "100bis109_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "110bis119_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "120bis129_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "130bis139_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "140bis149_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "150bis159_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "160bis169_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "170bis179_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
        "180undmehr_Flaeche_der_Wohnung_10m2_Intervalle_10km-Gitter",
    ]

    # --- Geburtsland (groups) ---
    gl_tot_10 = "Insgesamt_Bevoelkerung_Geburtsland_Gruppen_10km-Gitter"
    gl_cats_10 = [
        "Deutschland_Geburtsland_Gruppen_10km-Gitter",
        "Ausland_Sonstige_Geburtsland_Gruppen_10km-Gitter",
        "EU27_Land_Geburtsland_Gruppen_10km-Gitter",
        "Sonstiges_Europa_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Welt_Geburtsland_Gruppen_10km-Gitter",
        "Sonstige_Geburtsland_Gruppen_10km-Gitter",
    ]

    def topic(name, tot_10, cats_10, alpha, blend):
        if level == "1km":
            parent_cols = cats_10
            child_cols  = levelize(cats_10, "1km")
            child_total = tot_10.replace("_10km-Gitter", "_1km-Gitter")
        elif level == "100m":
            parent_cols = levelize(cats_10, "1km")
            child_cols  = levelize(cats_10, "100m")
            child_total = tot_10.replace("_10km-Gitter", "_100m-Gitter")
        else:
            raise ValueError(f"Unknown level: {level}")

        return TopicSpec(
            name=name,
            parent_cat_cols=parent_cols,
            child_cat_cols=child_cols,
            child_row_total_col=child_total,
            alpha=alpha,
            blend=blend,
        )

    specs = [
        topic("Familienstand",    fam_tot_10, fam_cats_10, alpha=0.90, blend=BLEND_STD),
        topic("Energietraeger",   et_tot_10,  et_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Heizungsart",      hz_tot_10,  hz_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Haushaltsgroesse", hh_tot_10,  hh_cats_10,  alpha=0.90, blend=BLEND_STD),
        topic("Lebensform",       lf_tot_10,  lf_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Raeume",           rm_tot_10,  rm_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Wohnflaeche",      fl_tot_10,  fl_cats_10,  alpha=0.85, blend=BLEND_STD),
        topic("Geburtsland",      gl_tot_10,  gl_cats_10,  alpha=0.85, blend=BLEND_STD),
    ]
    return specs


# ---------------------------------------------------------------------
# 5.5) Shim: create *_adj child totals for ALL topics and flip specs
# ---------------------------------------------------------------------

def _parent_total_name_for_child_total(child_total_col: str) -> str:
    """Swap suffix to get the corresponding parent level total name."""
    if child_total_col.endswith("_1km-Gitter"):
        return child_total_col.replace("_1km-Gitter", "_10km-Gitter")
    elif child_total_col.endswith("_100m-Gitter"):
        return child_total_col.replace("_100m-Gitter", "_1km-Gitter")
    else:
        return child_total_col

def apply_adj_for_all_topics(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    specs: List[TopicSpec],
    verbose: bool = True,
) -> List[TopicSpec]:
    """For each TopicSpec, create a *_adj child row-total and switch the spec to use it."""
    specs_out: List[TopicSpec] = []
    for spec in specs:
        total_col = spec.child_row_total_col
        adj_col = f"{total_col}_adj"
        if adj_col not in child_df.columns:
            parent_total = _parent_total_name_for_child_total(total_col)
            make_child_totals_adj(
                parent_df=parent_df,
                child_df=child_df,
                parent_id_col=parent_id_col,
                child_parent_id_col=child_parent_id_col,
                parent_adj_col=parent_total,
                child_total_col=total_col,
                out_col=adj_col,
            )
            if verbose:
                print(f"[adj] Created {adj_col} for topic '{spec.name}'")
        specs_out.append(TopicSpec(
            name=spec.name,
            parent_cat_cols=spec.parent_cat_cols,
            child_cat_cols=spec.child_cat_cols,
            child_row_total_col=adj_col,
            alpha=spec.alpha,
            blend=spec.blend,
        ))
    return specs_out


# ---------------------------------------------------------------------
# 6) Example run
# ---------------------------------------------------------------------

if __name__ == "__main__":
    # Paths
    PATH_10  = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df10_with_single_years.pickle"
    PATH_1   = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df1_with_single_years_100_4.pickle"
    PATH_100 = r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gender_backfilled_100_4.parquet"

    OUT_1    = r"C:\Users\petre\PycharmProjects\cleancensus\with_other_binned_data\cells_1km_with_binneds_100_4.parquet"
    OUT_100  = r"C:\Users\petre\PycharmProjects\cleancensus\with_other_binned_data\cells_100m_with_gender_backf_binneds_100_4.parquet"

    # Load pickles (reset index so IDs become normal cols)
    df10 = pd.read_pickle(PATH_10).reset_index(drop=False)
    df1  = pd.read_pickle(PATH_1).reset_index(drop=False)

    # Clean NaNs/±inf
    for df in (df10, df1):
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.fillna(0, inplace=True)

    # Normalize IDs
    df10["GITTER_ID_10km"] = df10["GITTER_ID_10km"].astype(str).str.strip()
    df1["GITTER_ID_10km"]  = df1["GITTER_ID_10km"].astype(str).str.strip()
    df1["GITTER_ID_1km"]   = df1["GITTER_ID_1km"].astype(str).str.strip()

    # -------------------------
    # 10km → 1km
    # -------------------------
    topics_1km = build_topic_specs_for_level("1km")

    # Normalize 10km category vectors to their *Insgesamt* totals, needed bcs they must match, but very often don't bcs of perturbation
    # (bit weird function naming. Parent just means that later 10km will be parent of 1km)
    normalize_parent_categories_for_specs(
        parent_df=df10,
        specs=topics_1km,
        child_level="1km",
        verbose=True
    )

    topics_1km = apply_adj_for_all_topics(
        parent_df=df10, child_df=df1,
        parent_id_col="GITTER_ID_10km", child_parent_id_col="GITTER_ID_10km",
        specs=topics_1km, verbose=True
    )


    for spec in tqdm(topics_1km, desc="Topics 1km"):
        res = downscale_topic(
            parent_df=df10, child_df=df1,
            parent_id_col="GITTER_ID_10km",
            child_parent_id_col="GITTER_ID_10km",
            spec=spec,
            inner_passes=10, outer_iters=2,
            rake_tol=1e-11, rake_max_iter=100,
            validate_row_tol=2e-4, verbose=False
        )
        for c in spec.child_cat_cols:
            df1[c] = res[c].values

    df1.to_parquet(OUT_1, index=False)

    # -------------------------
    # 1km → 100m (memory-safe, full-output)
    # -------------------------
    topics_100m = build_topic_specs_for_level("100m")

    # Build minimal column set needed to compute topics at 100m
    needed_cols_100 = {"GITTER_ID_1km"}
    for spec in topics_100m:
        needed_cols_100.add(spec.child_row_total_col)
        needed_cols_100.update(spec.child_cat_cols)
    needed_cols_100 = sorted(needed_cols_100)

    # Load ONLY needed columns (keeps original row order) and reset index
    df100_min = pd.read_parquet(PATH_100, columns=needed_cols_100).reset_index(drop=False)

    # Clean + downcast for RAM
    df100_min.replace([np.inf, -np.inf], np.nan, inplace=True)
    df100_min.fillna(0, inplace=True)
    for c in df100_min.columns:
        if pd.api.types.is_float_dtype(df100_min[c]):
            df100_min[c] = df100_min[c].astype(np.float32)
    df100_min["GITTER_ID_1km"] = df100_min["GITTER_ID_1km"].astype(str).str.strip()

    # link coverage flags
    p_1km = set(df1["GITTER_ID_1km"].unique())
    df100_min["is_orphan"] = ~df100_min["GITTER_ID_1km"].isin(p_1km)
    df100_ok = df100_min.loc[~df100_min["is_orphan"]].copy()
    df1_ok   = df1.loc[df1["GITTER_ID_1km"].isin(df100_ok["GITTER_ID_1km"])].copy()

    # Create *_adj and flip specs to use them
    topics_100m = apply_adj_for_all_topics(
        parent_df=df1_ok, child_df=df100_ok,
        parent_id_col="GITTER_ID_1km", child_parent_id_col="GITTER_ID_1km",
        specs=topics_100m, verbose=True
    )

    # Compute downscaled categories and place back into df100_min at matching positions
    for spec in tqdm(topics_100m, desc="Topics 100m"):
        res = downscale_topic(
            parent_df=df1_ok, child_df=df100_ok,
            parent_id_col="GITTER_ID_1km",
            child_parent_id_col="GITTER_ID_1km",
            spec=spec,
            inner_passes=10, outer_iters=2,
            rake_tol=1e-11, rake_max_iter=100,
            validate_row_tol=2e-4, verbose=False
        )
        for c in spec.child_cat_cols:
            df100_min.loc[df100_ok.index, c] = res[c].values.astype(np.float32)

    # --- Write FULL output parquet: stream original + append new columns by position ---
    dataset = ds.dataset(PATH_100, format="parquet")
    new_cols = [c for spec in topics_100m for c in spec.child_cat_cols]
    # ensure all new cols exist in df100_min (orphans keep NaN/0 as you prefer)
    for c in new_cols:
        if c not in df100_min.columns:
            df100_min[c] = np.nan

    # Prepare writer schema = original + any extra fields
    orig_schema = dataset.schema
    extra_fields = [pa.field(c, pa.float32()) for c in new_cols if c not in orig_schema.names]
    full_schema = pa.schema(list(orig_schema) + extra_fields)

    writer = None
    pos = 0
    batch_size = 1_000_000  # tune if needed

    scan = dataset.scan(columns=None, batch_size=batch_size)
    for rb in scan.to_reader():
        tbl = rb.to_table()
        n = tbl.num_rows

        # build Arrow arrays for appended columns from df100_min slice
        appended = {}
        for c in new_cols:
            arr = pa.array(df100_min[c].iloc[pos:pos+n].to_numpy(), type=pa.float32())
            appended[c] = arr

        combined = tbl
        for c, arr in appended.items():
            if c in combined.schema.names:
                combined = combined.set_column(combined.schema.get_field_index(c), c, arr)
            else:
                combined = combined.append_column(c, arr)

        if writer is None:
            writer = pq.ParquetWriter(OUT_100, combined.schema)
        writer.write_table(combined)
        pos += n

    if writer:
        writer.close()

    # -------------------------
    # quick echoes (use the small frames)
    # -------------------------
    def _echo(df, cols, total_col, tag):
        if total_col not in df.columns:
            print(f"[{tag}] skipped: total column '{total_col}' not found.")
            return
        calc = df[cols].sum(axis=1).to_numpy(float)
        given = pd.to_numeric(df[total_col], errors="coerce").fillna(0.0).to_numpy(float)
        rel = np.abs(calc - given) / np.maximum(given, 1.0)
        print(f"[{tag}] rows={len(df):,} | max row rel.err={rel.max():.3e} | mean={rel.mean():.3e}")

    # Echo (Familienstand)
    fam1_cols = levelize([
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ], "1km")
    _echo(df1, fam1_cols,
          "Insgesamt_Bevoelkerung_Familienstand_1km-Gitter_adj",
          "1km Familienstand")

    fam100_cols = levelize([
        "Ledig_Familienstand_10km-Gitter",
        "Verheiratet_Familienstand_10km-Gitter",
        "Verwitwet_Familienstand_10km-Gitter",
        "Geschieden_Familienstand_10km-Gitter",
        "EingetrLebenspartnerschaft_Familienstand_10km-Gitter",
        "EingetrLebenspartVerstorben_Familienstand_10km-Gitter",
        "EingetrLebenspartAufgehoben_Familienstand_10km-Gitter",
        "OhneAngabe_Familienstand_10km-Gitter",
    ], "100m")
    # echo on the small frame (df100_min) which now holds the computed columns
    _echo(df100_min, fam100_cols,
          "Insgesamt_Bevoelkerung_Familienstand_100m-Gitter_adj",
          "100m Familienstand")


[norm] 'Familienstand' rows=3,824 | max rel.err=4.76e-16 | mean=7.80e-17
[norm] 'Energietraeger' rows=3,824 | max rel.err=5.67e-16 | mean=8.08e-17
[norm] 'Heizungsart' rows=3,824 | max rel.err=3.86e-16 | mean=5.16e-17
[norm] 'Haushaltsgroesse' rows=3,824 | max rel.err=3.97e-16 | mean=6.62e-17
[norm] 'Lebensform' rows=3,824 | max rel.err=4.10e-16 | mean=6.77e-17
[norm] 'Raeume' rows=3,824 | max rel.err=3.93e-16 | mean=3.77e-17
[norm] 'Wohnflaeche' rows=3,824 | max rel.err=5.87e-16 | mean=8.18e-17
[norm] 'Geburtsland' rows=3,824 | max rel.err=4.19e-16 | mean=7.26e-17
[adj] Created Insgesamt_Bevoelkerung_Familienstand_1km-Gitter_adj for topic 'Familienstand'
[adj] Created Insgesamt_Energietraeger_Energietraeger_1km-Gitter_adj for topic 'Energietraeger'
[adj] Created Insgesamt_Heizungsart_Gebaeude_nach_ueberwiegender_Heizungsart_1km-Gitter_adj for topic 'Heizungsart'
[adj] Created Insgesamt_Haushalte_Groesse_des_privaten_Haushalts_1km-Gitter_adj for topic 'Haushaltsgroesse'
[adj] Created I

Topics 1km:  12%|█▎        | 1/8 [00:45<05:17, 45.37s/it]



[DEBUG mismatch]
Parent ID: CRS3035RES10000mN3190000E4530000
Parent totals (p):
[134.17713153 554.73231989 191.25247852   0.          64.08460013
  98.12954395  23.03040317 436.5763384   13.0171844 ]
Child sums (X.sum(axis=0)):
[134.16272388 554.67222941 191.23224394   0.          64.07766596
  98.11896845  23.02796206 436.69238031  13.01582599]
Difference (child - parent):
[-0.01440765 -0.06009049 -0.02023458  0.         -0.00693417 -0.01057551
 -0.00244111  0.11604191 -0.00135841]
Relative difference:
[-0.00010738 -0.00010832 -0.0001058   0.         -0.0001082  -0.00010777
 -0.000106    0.0002658  -0.00010435]


Topics 1km:  50%|█████     | 4/8 [02:50<02:40, 40.05s/it]



[DEBUG mismatch]
Parent ID: CRS3035RES10000mN3540000E4220000
Parent totals (p):
[424.2150838  188.20810056  53.77374302  22.90363128  19.91620112
   3.98324022]
Child sums (X.sum(axis=0)):
[424.23410671 188.21643264  53.7761513   22.90465733  19.91697439
   3.95167764]
Difference (child - parent):
[ 0.01902291  0.00833208  0.00240828  0.00102604  0.00077327 -0.03156258]
Relative difference:
[ 4.48426054e-05  4.42705533e-05  4.47854934e-05  4.47981763e-05
  3.88262882e-05 -7.92384623e-03]


Topics 1km:  75%|███████▌  | 6/8 [04:10<01:20, 40.04s/it]



[DEBUG mismatch]
Parent ID: CRS3035RES10000mN2980000E4500000
Parent totals (p):
[ 0.          2.9877551   7.96734694 10.95510204 18.92244898 15.93469388
 22.90612245 24.89795918 26.88979592 15.93469388 30.87346939  8.96326531
 13.94285714 12.94693878 10.95510204  2.9877551  15.93469388]
Child sums (X.sum(axis=0)):
[ 0.          2.94489952  7.96876365 10.95705002 18.92581367 15.9375273
 22.9101955  24.90238641 26.89457733 15.9375273  30.87895915  8.96485911
 13.94533639 12.94924093 10.95705002  2.98828637 15.9375273 ]
Difference (child - parent):
[ 0.         -0.04285558  0.00141671  0.00194798  0.00336469  0.00283343
  0.00407305  0.00442723  0.00478141  0.00283343  0.00548976  0.0015938
  0.00247925  0.00230216  0.00194798  0.00053127  0.00283343]
Relative difference:
[ 0.         -0.01434374  0.00017781  0.00017781  0.00017781  0.00017781
  0.00017781  0.00017781  0.00017781  0.00017781  0.00017781  0.00017781
  0.00017781  0.00017781  0.00017781  0.00017781  0.00017781]


Topics 1km: 100%|██████████| 8/8 [06:46<00:00, 50.79s/it]


ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column werterlaeuternde_Zeichen_Anteil_Auslaender_1km-Gitter with type object')